# Cantilever Snap-Fit Design Calculator
**Plastic Design Calculators — Notebook 3**

---

## Part 1: Theory and Governing Equations

### 1.1 Snap-Fit Mechanics

A cantilever snap-fit is a flexible beam that deflects laterally during assembly to bypass a locking undercut. Two competing requirements must be balanced:
1. **Flexibility**: beam must deflect through the undercut without yielding.
2. **Stiffness**: assembled joint must resist separation forces in service.

### 1.2 Root Strain

Maximum strain occurs at the **root** of the cantilever:

$$\boxed{\epsilon_{\max} = \frac{1.5 \cdot t \cdot Y}{L^2 \cdot Q}}$$

- $t$ — beam thickness at root [m]
- $Y$ — required deflection to clear undercut [m]
- $L$ — effective beam length [m]
- $Q$ — wall compliance factor: $Q = 1$ for rigid wall, $Q > 1$ for compliant wall

### 1.3 Allowable Strain (repeated assembly)

For parts requiring repeated snap-fitting, the allowable strain is de-rated:

$$\epsilon_{\mathrm{allow}} = 0.6 \times 0.7 \times \epsilon_{\mathrm{yield}}$$

### 1.4 Deflection Force

Theoretical perpendicular force to deflect the beam:

$$P = \frac{b \cdot t^2 \cdot E \cdot \epsilon_{\max}}{6 \cdot L}$$

where $b$ is the beam width and $E$ is the secant flexural modulus at the applicable strain.

### 1.5 Mating (Assembly) Force

Friction on the angled lead-in surface amplifies the deflection force:

$$\boxed{W_{\mathrm{assemble}} = P \cdot \frac{\mu + \tan\alpha}{1 - \mu\tan\alpha}}$$

For **retention** (separation) force, substitute the return angle $\beta$ for $\alpha$:

$$W_{\mathrm{retain}} = P \cdot \frac{\mu + \tan\beta}{1 - \mu\tan\beta}$$

> **Note:** When $\beta \to 90°$, the denominator $\to 0$ and the joint becomes **inseparable** (permanent snap-fit).

### 1.6 Half-Circular Cross-Section Deflection Limit

$$Y_{\max} = 0.578 \cdot \epsilon \cdot \frac{L^2}{r}$$

where $r$ is the cross-section radius.

### Assumptions
- Classical Euler–Bernoulli beam theory (small deflection)
- Uniform rectangular cross-section (unless geometric abstraction mode is used)
- Secant modulus approximation is valid for strains up to yield
- **Warning:** If $Y/L > 0.2$, nonlinear FEA is recommended

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import FRICTION_COEFFICIENTS

# ── Geometry ──────────────────────────────────────────────────────────────────
t_root  = Q_(2.0,  'mm')    # beam thickness at root
b_width = Q_(8.0,  'mm')    # beam width
L_beam  = Q_(20.0, 'mm')    # effective beam length
Y_defl  = Q_(1.5,  'mm')    # required deflection (undercut height)
Q_wall  = 1.0               # wall compliance factor (1 = rigid)

# ── Angles ────────────────────────────────────────────────────────────────────
alpha_deg = 30.0   # lead-in angle [degrees]
beta_deg  = 60.0   # return (retention) angle [degrees]

# ── Material properties ───────────────────────────────────────────────────────
E_secant   = Q_(1_300.0, 'MPa')   # secant flexural modulus at operating strain
eps_yield  = 0.10                 # yield strain [-] (e.g. 10% for PP)
mu_frict   = FRICTION_COEFFICIENTS['PP_steel']

# ── Advanced: geometric abstraction mode ─────────────────────────────────────
# For non-rectangular cross-sections, supply I and c directly.
# Set USE_CUSTOM_SECTION = True and provide values below.
USE_CUSTOM_SECTION = False
I_custom = Q_(None, 'mm**4')  # area moment of inertia
c_custom = Q_(None, 'mm')     # neutral axis distance

print(f"μ friction (PP/steel) : {mu_frict}")
print(f"Yield strain          : {eps_yield*100:.1f} %")
print(f"Deflection ratio Y/L  : {strip_units(Y_defl.to('mm'))/strip_units(L_beam.to('mm')):.3f}")

---
## Part 3: Computation Engine

In [ ]:
# ── 3.1  Symbolic equations ───────────────────────────────────────────────────
t_s, b_s, L_s, Y_s, Q_s = sp.symbols('t b L Y Q', positive=True)
E_s, eps_s, mu_s         = sp.symbols('E epsilon mu', positive=True)
alpha_s, beta_s, P_s     = sp.symbols('alpha beta P', positive=True)

eps_max_eq = sp.Rational(3,2) * t_s * Y_s / (L_s**2 * Q_s)
P_eq       = b_s * t_s**2 * E_s * eps_s / (6 * L_s)
W_eq       = P_s * (mu_s + sp.tan(alpha_s)) / (1 - mu_s * sp.tan(alpha_s))

print("Root strain ε_max =")
sp.pprint(eps_max_eq)
print()
print("Deflection force P =")
sp.pprint(P_eq)
print()
print("Mating force W =")
sp.pprint(W_eq)

In [ ]:
def root_strain(t_m, Y_m, L_m, Q=1.0, **kwargs):
    """Maximum root strain of a cantilever snap-fit beam.

    Args:
        t_m (float): Root thickness [m].
        Y_m (float): Required deflection [m].
        L_m (float): Effective beam length [m].
        Q (float): Wall compliance factor (default 1.0 = rigid wall).
        **kwargs: Reserved for tapered-beam correction factors.

    Returns:
        float: Maximum strain εmax [-].
    """
    return 1.5 * t_m * Y_m / (L_m**2 * Q)


def deflection_force(b_m, t_m, E_Pa, eps_max, L_m, **kwargs):
    """Perpendicular deflection force for a rectangular snap-fit beam.

    Args:
        b_m (float): Beam width [m].
        t_m (float): Root thickness [m].
        E_Pa (float): Secant flexural modulus [Pa].
        eps_max (float): Maximum root strain [-].
        L_m (float): Effective beam length [m].
        **kwargs: Accepts I_m4 and c_m for custom section (geometric abstraction).

    Returns:
        float: Deflection force P [N].
    """
    if kwargs.get('I_m4') is not None and kwargs.get('c_m') is not None:
        # Generic first-principles formulation: P = E*I*ε_max / (c * L)
        return E_Pa * kwargs['I_m4'] * eps_max / (kwargs['c_m'] * L_m)
    return b_m * t_m**2 * E_Pa * eps_max / (6.0 * L_m)


def mating_force(P_N, mu, angle_deg, **kwargs):
    """Actual assembly/retention force accounting for lead-in friction.

    Args:
        P_N (float): Perpendicular deflection force [N].
        mu (float): Coefficient of friction.
        angle_deg (float): Lead-in angle α or return angle β [degrees].
        **kwargs: Reserved for Hertz contact area corrections.

    Returns:
        float: Mating force W [N]. Returns np.inf if joint is inseparable.
    """
    alpha_rad = np.radians(angle_deg)
    denom = 1.0 - mu * np.tan(alpha_rad)
    if np.abs(denom) < 1e-6:
        return np.inf   # inseparable snap-fit
    return P_N * (mu + np.tan(alpha_rad)) / denom


print("Snap-fit functions defined.")

In [ ]:
# ── 3.2  Numerical evaluation ─────────────────────────────────────────────────
t_m = strip_units(t_root.to('meter'))
b_m = strip_units(b_width.to('meter'))
L_m = strip_units(L_beam.to('meter'))
Y_m = strip_units(Y_defl.to('meter'))
E_Pa = strip_units(E_secant.to('Pa'))

# Deflection ratio check
defl_ratio = Y_m / L_m
if defl_ratio > 0.20:
    print(f"⚠ WARNING: Y/L = {defl_ratio:.3f} > 0.20 — classical beam theory invalid. Consult nonlinear FEA.")

eps_max = root_strain(t_m, Y_m, L_m, Q_wall)
eps_allow = 0.6 * 0.7 * eps_yield

P_N  = deflection_force(b_m, t_m, E_Pa, eps_max, L_m)
W_a  = mating_force(P_N, mu_frict, alpha_deg)
W_r  = mating_force(P_N, mu_frict, beta_deg)
permanent_snap = np.isinf(W_r)

print(f"Root strain ε_max     : {eps_max*100:.3f} %")
print(f"Allowable strain      : {eps_allow*100:.3f} %")
print(f"Deflection force P    : {P_N:.2f} N")
print(f"Assembly force W_a    : {W_a:.2f} N  (α={alpha_deg}°)")
if permanent_snap:
    print(f"Retention force W_r   : INSEPARABLE JOINT  (β={beta_deg}° → permanent snap-fit)")
else:
    print(f"Retention force W_r   : {W_r:.2f} N  (β={beta_deg}°)")

# Parametric sweep: strain vs beam length (vectorised)
L_sweep = np.linspace(5e-3, 40e-3, 300)
eps_sweep = root_strain(t_m, Y_m, L_sweep, Q_wall)
P_sweep   = deflection_force(b_m, t_m, E_Pa, eps_sweep, L_sweep)

# Force vs lead angle (vectorised)
alpha_sweep = np.linspace(5.0, 85.0, 300)
W_alpha     = np.array([mating_force(P_N, mu_frict, a) for a in alpha_sweep])

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Root strain vs beam length ────────────────────────────────────────
ax1 = axes[0]
ax1.plot(L_sweep * 1e3, eps_sweep * 100, color='steelblue', lw=2, label='ε_max')
ax1.axhline(eps_allow * 100, color='tomato',    ls='--', lw=1.5, label=f'ε_allow = {eps_allow*100:.2f} %')
ax1.axhline(eps_yield * 100, color='goldenrod', ls='--', lw=1.5, label=f'ε_yield = {eps_yield*100:.1f} %')
ax1.axvline(L_m * 1e3,       color='purple',    ls=':',  lw=1.5, label=f'Design L = {L_m*1e3:.0f} mm')

# shade unsafe region
ax1.fill_between(L_sweep * 1e3, eps_allow * 100, eps_sweep * 100,
                 where=eps_sweep > eps_allow, alpha=0.2, color='tomato', label='Strain exceeded')
ax1.set_xlabel('Beam Length L [mm]')
ax1.set_ylabel('Root Strain ε_max [%]')
ax1.set_title('Root Strain vs Beam Length\n(t={:.1f} mm, Y={:.1f} mm)'.format(t_m*1e3, Y_m*1e3))
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Mating force vs lead angle ────────────────────────────────────────
ax2 = axes[1]
# clip inf values for display
W_display = np.where(np.isfinite(W_alpha), W_alpha, np.nan)
ax2.plot(alpha_sweep, W_display, color='darkorange', lw=2)
ax2.axvline(alpha_deg, color='steelblue', ls='--', lw=1.5, label=f'Lead angle α={alpha_deg}°')
ax2.axvline(beta_deg,  color='tomato',    ls='--', lw=1.5, label=f'Return angle β={beta_deg}°')
ax2.set_xlabel('Angle [degrees]')
ax2.set_ylabel('Force W [N]')
ax2.set_title(f'Mating / Retention Force vs Angle\n(μ={mu_frict}, P={P_N:.1f} N)')
ax2.set_ylim(0, min(3 * W_a, 500))
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.annotate('Inseparable →', xy=(85, ax2.get_ylim()[1]*0.85), ha='right', fontsize=8, color='tomato')

plt.tight_layout()
plt.savefig('03_snap_fit_output.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Design Rule Validation

In [ ]:
pass_strain  = eps_max <= eps_allow
pass_defl    = defl_ratio <= 0.20
pass_perm    = True   # permanent snap-fit is valid if intentional

def badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

print("═" * 62)
print("  DESIGN RULE VALIDATION — CANTILEVER SNAP-FIT")
print("═" * 62)
print(f"  Root strain vs allowable:")
print(f"    ε_max   = {eps_max*100:.3f} %")
print(f"    ε_allow = {eps_allow*100:.3f} %  (0.6 × 0.7 × ε_yield)")
print(f"    Result  : {badge(pass_strain)}")
print()
print(f"  Classical beam theory validity (Y/L ≤ 0.20):")
print(f"    Y/L     = {defl_ratio:.3f}")
print(f"    Result  : {badge(pass_defl)}")
if not pass_defl:
    print(f"    ⚠ Nonlinear FEA required for accurate force prediction.")
print()
if permanent_snap:
    print(f"  Joint classification: PERMANENT (inseparable) snap-fit at β={beta_deg}°")
else:
    print(f"  Retention force W_r = {W_r:.2f} N  → detachable snap-fit")
print("═" * 62)
overall = pass_strain
if overall:
    print("  ✓ OVERALL: DESIGN PASSES strain criteria.")
else:
    print("  ✗ OVERALL: DESIGN FAILS — increase L or reduce t/Y.")
print("═" * 62)